# Lecture 09 — Functions, cont.
### EES 3350/5350 · Python in Earth Science · Wednesday, September 15, 2026

**Learning objectives for today**
- recognize when repeating a task calls for a loop instead of copy-pasting code
- write a `while` loop that repeates as long as a condition is `True`, and recognize (and kill) an infinite loop
- write a `for` loop that iterates over each item in a list

---
## Part 0 : setting up functions

In [1]:
import math

def factor_of_safety(cohesion, soil_depth, soil_density, 
                     slope_deg, friction_deg, rw,  gravity=9.81, water_density=1000):
    """
    Compute the infinite slope Factor of Safety (FS).

    Parameters
    ----------
    cohesion : float
        Cr + Cs : cohesion - bonds holding roots (r) and soil (s) together (Pa)
    soil_depth : float
        depth of soil perpendicular to the slope (m)
    soil_density : float
        density of wet soil (kg/m^3)
    gravity : float
        gravitational acceleration (m/s^2)
    slope_deg : float
        slope angle (angle of the soil surface from horizontal) (degrees)
    friction_deg : float
        internal friction angle of the soil (degrees)
    rw : float
        relative wetness - ratio of groundwater depth to soil depth (1 = fully saturated) (0-1)
    water_density : float
        density of water (kg/m^3)

    Returns
    -------
    float, or None if slope_deg == 0 (undefined, would divide by 0)
    """
    # converting degrees to radians
    theta = math.radians(slope_deg)
    phi = math.radians(friction_deg)

    # A
    denom_A = soil_depth * soil_density * gravity * math.sin(theta)
    A = cohesion/denom_A
    # B
    parenth_B = (1 - ((rw * water_density) / soil_density))
    numerat_B = math.cos(theta) * math.tan(phi) * parenth_B
    B = numerat_B / math.sin(theta)

    FS = A + B
    
    return FS

def interpret_stability(fs):
    """Classify a Factor of Safety value into a stability category."""
    if fs < 1.0:
        return "Unstable"
    elif fs < 1.3:
        return "Marginally stable"
    else:
        return "Stable"

##### Remember **Exercise 8 from lab 3**? You held everything fixed expect coehsion, and had to compute FS for six different cohesion values:

In [ ]:
cohesion_values = [0, 4000, 8000, 12000, 26000, 20000]

fs_cohesion = [
    factor_of_safety(cohesion_values[0], 1.0, 2000, 25, 32, 0.5),
    factor_of_safety(cohesion_values[1], 1.0, 2000, 25, 32, 0.5),
    factor_of_safety(cohesion_values[2], 1.0, 2000, 25, 32, 0.5),
    factor_of_safety(cohesion_values[3], 1.0, 2000, 25, 32, 0.5),
    factor_of_safety(cohesion_values[4], 1.0, 2000, 25, 32, 0.5),
    factor_of_safety(cohesion_values[5], 1.0, 2000, 25, 32, 0.5),
]

print(fs_cohesion)

#### Look familiar? Did you hate writing all of that out that many times?
You had to type out `factor_of_safety()` six times, changing one index each time. That's fine for 6 values, but
#### What if you had 50 cohesion values instead of 6? What if you had 500?
That's exactly the problem loops solve: doing the same thing *over and over*, without writing it out *over and over*. Today we'll learn two ways to repeat code:
- `while` loops
- `for` loops
And then we'll come back and fix the cell above.

---
## Part 1 : `while` loops
A `while` loop repeats a block of code **as long as a condition stays `True`**. It's especially useful when you don't know ahead of time exactly how many repetitions you'll need. 
> For example, we could execute something liie "keep raining until the slope fails", rather than "do this exactly 6 times".

#### Let's start simple : **a storm is raising the relative wetness (`rw`) of a hillslope over time.**
We don't know in advance how many steps it'll take to reach full saturation, we just know it keeps going *while* `rw` is below 1.0.

<div class="alert alert-info">

### Exercise 1
The code cell below uses a `while` loop and things you've learned so far in class. With a neighbor, add comments to each line with **what you think is happening**.
    
</div>

In [ ]:
# your answer here
rw = 0.0             # []
rw_increase = 0.1    # []

while rw < 1.0:               # []
    print(f"Rw = {rw:.1f}")   # []
    rw += rw_increase         # []

print("Soil is fully saturated.") 

### The new things:

#### **The `while` statement**

Similar to `if`, `elif`, `else`, and `def`, while has straightforward syntax:

| Piece | What it means | In cell above |
|---|---|---|
| **`while`** | keyword that starts the repetition (turns green in Jupyter | `while` |
| **condition** | whatever follows `while`; the loop keeps going as loing as this is `True` | `rw < 1.0` |
| **colon** | tells Python that the indented code below belongs to the loop | `:` |
|**indented content**| the code that repeats each pass through the loop | `print(f"Rw = {rw:.1f}")` and `rw += rw_increase`|
|**something that changes**| *you* are responsible for updating whatever the condition checks; nothing does this automatically | `rw += rw_increase` |

That last row matters more than it looks.

### Common Mistake: infinite loops
`while` will keep checking its condition **forever** unless something inside the loop eventually makes it `False`. If you forget to update the variable the condition depends on, the loop never ends!


#### Let's see what happens when we forget that update line
```python
rw = 0.0
while rw < 1.0:
    print(f"RW = {rw}")
    # forgot to increase the rw here!
```
This would print `Rw = 0.0` forever.

> **If this happens to you: click the square () "interrupt the kernal" button in the toolbar.
> You'll be left with a wall of output. Right click the cell and choose **Clear Cell Outp7ut** (or Clear Outputs) to clean it up.

Try it yourself!

In [ ]:
rw = 0.0
while rw < 1.0:
    print(f"RW = {rw}")
    # forgot to increase the rw here!

<div class="alert alert-info">

### Exercise 2
A storm arrives at a marginal hillslope:
- cohesion = 2000
- soil_depth = 1.0
- soil_density = 2000
- slope = 32 deg
- friction_deg = 30

Write a `while` loop that:
- starts `rw` at 0.0 and increases it by 0.05 each step
- at each step, calls `factor_of_safety()` with the values above and the current `rw`
- prints the current `rw` (to 2 decimal places) and the resulting FS (to three decimal places)
- stops the loop once FS drops below 1.0  (i.e., the slope has failed.
    
</div>

<div class="alert alert-success">

**Tip:** If you want to exit a loop when a condition is met, you can use the keyword `break`.

> **`break`** will immediately exit the loop the moment the condition inside is met.

This is handy if we want to stop at a certain place. We'll learn more about `break` later this week.
    
</div>

In [ ]:
# your answer here
rw = ________ # initial rw value

while ______:
    ________ # use of factor_of_safety
    ________ # print statement(s)
    if fs < 1.0:
        break
    rw += ______

---
## Part 2 : `for` loops
A `for` loop is for cases where you want to do something for each item in a collection, like a list. Instead of a condition that could run any number of times, a `for` loop runs once per item, and then it's done.

In [ ]:
rw_values = [0.0, 0.25, 0.5, 0.75, 1.0]

for rw in rw_values:
    print(f"Rw = {rw}")

### The new things:

#### **The `for` statement**

Similar to `if`, `elif`, `else`, and `def`, while has straightforward syntax:

| Piece | What it means | In cell above |
|---|---|---|
| **`for`** | keyword that starts the iteration (turns green in Jupyter) | `for` |
| **loop variable** | a new variable that takes on each item's value, one at a time | `rw` |
| **in** | separates the loop variable from the collection being iterated | `in` |
|**collection**| the list (or other iterable) being looped over | `rw_values`|
|**colon + indented body**| same as `while` ; this is what repeates | `print(...)` |

The loop variable's name doesn't matter - `rw`, `x`, `value`, whatever you pick - as long as you use that same name inside the loop body.

#### Let's put this to work: the cohesion sweep from Lab 03
Instead of typing `factor_of_safety(...)` six times, we can loop over `cohesion_values`, computing FS for each one and appending it to a new list.

In [ ]:
cohesion_values = [0, 4000, 8000, 12000, 26000, 20000] #didn't technically need since we have this earlier too
fs_cohesion = []

for c in cohesion_values:
    fs = factor_of_safety(c, 1.0, 2000, 25, 32, 0.5)
    fs_cohesion.append(fs)

print(fs_cohesion)

the `fs_cohesion = []` is an empty array that is added onto during each loop. Notice how we use the `.append()` method.

<div class="alert alert-info">

### Exercise 3
Do the same thing for Lab 3 Exercise 9, the sweep of **friction angle**. Holding the following variables fixed, use a `for` loop to compute FS for each value in `friction_values`, appending each result to `fs_friction`.
- `cohesion` = 2000
- `soil_depth` = 1.0
-  soil density = 2000
- `slope_deg` = 25
- `rw` = 0.5
    
</div>

In [ ]:
# your answer here
friction_values = [15. 20, 35, 30, 35, 40, 45]
fs_friction = []

for ______ in ______:
    ______ # use of factor_of_safety
    ______ # add to fs_friction

prin(fs_friction)

#### A third way to build a list: indexing
So far, you've built `fs_cohesion` and `fs_friction` by starting with an empty list and usign `.append()` to grow it one item at a time. There's another way!

You could also write directly into a specific position (index) of a list tha already has slots for your results.
Let's try adapting the cohesion sweep this way.

### Spot the bug
Here's a first attempt at rewriting the cohesion sweep using indexing instead of `.append()`. Run it and see what happens.

In [ ]:
cohesion_values = [0, 4000, 8000, 12000, 26000, 20000] #didn't technically need since we have this earlier too
fs_cohesion = []

for c in cohesion_values:
    fs = factor_of_safety(c, 1.0, 2000, 25, 32, 0.5)
    fs_cohesion[c] = fs

print(fs_cohesion)

### Answer

`IndexError: list assignment out of range`. Two things went wrong here!

1. **`fs_cohesion` is still empty**. You cannot write to position `0` of a list that has no positions yet, any more than you can put a book on shelf #3 of a bookcase that has zero shelves.
2. **`c is a *value* from the list, not a *position* in the list**`. The first cohesion value is 0, so Python tries `fs_cohesion[0]`, but even if the list weren't empty, later on `c` would be `4000`, `8000`, `20000` ... those are cohesion values, not valid list positions. A `for c in cohesion_values` loop hands you each *item*, not it's index.

<div class="alert alert-success">

**Tip**: to index into a list, you need two things:
- a list that already has the right number of slots, and
- a counter that tracks *position* (0, 1, 2, ...) rather than the values themselves

One way to pre-make the slots:
- `[None]*6` makes a list of six placeholder `None` values you can fill in later.
    
</div>

In [ ]:
cohesion_values = [0, 4000, 8000, 12000, 26000, 20000] #didn't technically need since we have this earlier too
fs_cohesion = [None] * len(cohesion_values) # 6 empty placeholder slots

i = 0 # our position counter, separate from the cohesion values themselves

while i < len(cohesion_values): # note we changed this to a "while" loop
    c = cohesion_values[i]
    fs = factor_of_safety(c, 1.0, 2000, 25, 32, 0.5)
    fs_cohesion[i] = fs
    i += 1

print(fs_cohesion)

<div class="alert alert-info">

### Exercise 4

Rewrite Exercise 3 using indexing instead of `.append()`. Pre-allocate `fs_friction` with placeholder slots, then use a `while` loop with a position counter to fill it in.
    
</div>

In [ ]:
# your answer here
friction_values = [15, 20, 25, 30, 35, 40, 45]
fs_friction = ______

i = 0
while ______:
    ______
    ______
    ______
    ______

print(fs_friction)

## Extra Practice
Start these in class and finish them for homework.

<div class="alert alert-info">

### Exercise 5

Using the "medium" scenario values and holding `rw=0.5` fixed, write a `for` loop over `slope_angles` below that for each slope, prints a sentence giving the slope, the FS value, and its stability classification from `interprety_stability()`
- `cohesion=8500`
- `soil_depth=1.0`
- `soil_density=2000`
- `friction_deg=32`

</div>

In [ ]:
# your answer here
slope_angles = [10, 20, 30, 40, 50]

for s in slope_angles:
    fs = ______
    category = ______
    print(f"At a slope of {s} degrees, FS = {fs:.3f} -> {category}")


<div class="alert alert-info">

### Exercise 6

Soil slowly builds up over time as bedrock weathers. Write a `while` loop that starts `soil_depth` at `0.2` m and increases it by `0.1` per "step", printing the step number and current depth each time, and stops once `soil_depth` reaches or exceeds `2.0` m. After the loop, print how many steps it took.

</div>

In [ ]:
# your answer here
soil_depth = ______
step = 0

while ______:
    print(f"step {step}: soil depth = {soil_depth:.2f} m")
    soil_depth += ______
    step += ______

print(f"exceeded 2.0 m after {step} steps")


**Challenge:** How would we be sure to print the last step instead of ending at 1.9 m?

<div class="alert alert-info">

### Exercise 7

Storms don't always rain at a steady rate. Using the same marginal hillslope from Exercise (below), write a `while` loop where `rw` starts at `0.0` and increases each step, BUT the storm intensifies partway through:
- while `rw` is below `0.15`, increase `rw` by `0.03` per step (a light early rain)
- once `rw` reaches `0.15` or higher, increase `rw` by `0.08` per step instead (the storm really gets going)

At each step, print the step number, `rw`, and FS, and stop the loop (and print a failure message) once FS drops below `1.0`. This means you'll need an `if`/`else` *inside* your `while` loop to decide which increase amount to use

</div>

In [ ]:
# your answer here